# Deep Learning-Based Lung Sound Event Detection
## An Evaluation of Class-Imbalance-Aware Loss Functions

Supporting notebook for the MSc dissertation. It implements a CNN-BiGRU frame-level
respiratory sound event detector on HF_Lung_V1 and compares four loss configurations
(standard BCE, raw class-weighted BCE, capped class-weighted BCE, and Focal Loss) for
continuous (CAS) and discontinuous (DAS) adventitious sound detection.

**Expected Google Drive layout**

```
MyDrive/HF_Lung_V1-master/
    train/train/        raw .wav files and *_label.txt annotations (training partition)
    test/test/          raw .wav files and *_label.txt annotations (test partition)
    processed/train/    cached .npz feature files, written by Section 2
    processed/test/     cached .npz feature files, written by Section 2
    models/             single-run checkpoints
    models/seeds/       per-seed checkpoints for the multi-seed analysis
```

**How to run**

1. Run Section 1 (mount Drive) and Section 2 (definitions) in every new session; they only
   define functions and cost nothing.
2. Section 2 caches features to `processed/`. Files that already exist are skipped, so the
   cell is safe to re-run.
3. All training functions reload an existing checkpoint instead of retraining unless
   `force_retrain=True` is passed, so the published results can be reproduced without a GPU
   run. The first call is slow because the training directory is rescanned over Drive.
4. Sections 7, 8 and 9 produce Tables I-II, Table III and Table IV of the paper respectively.

**Test-set protocol.** The predefined test partition is read only in Sections 7, 8 and 9,
and only for final evaluation. Early stopping, learning-rate scheduling, checkpoint
selection, cap selection and threshold selection all use a validation split carved out of
the training data.

Requirements: Python 3, PyTorch, librosa, scipy, scikit-learn, pandas, tqdm (all
preinstalled in Google Colab). Training was carried out on an NVIDIA A100 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Environment setup

Mount Google Drive, which holds the dataset, the cached features and all model checkpoints.

In [ ]:
import os, re, numpy as np, torch, torch.nn as nn, librosa
from collections import defaultdict
from scipy.signal import butter, sosfiltfilt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from tqdm import tqdm

_HPF_SOS = butter(N=10, Wn=80, btype='highpass', fs=4000, output='sos')

def preprocess(wav_path, sr=4000):
    """Convert one .wav recording into its frame-level feature matrix (938 x 193).

    Pipeline (Section III.B of the paper):
      1. Load the waveform at 4 kHz.
      2. Apply a 10th-order Butterworth 80 Hz high-pass filter. sosfiltfilt performs
         zero-phase forward-backward filtering, so no group delay is introduced.
      3. STFT with a Hann window, n_fft=256, hop_length=64 -> 938 frames per 15 s file.
      4. Concatenate three feature groups per frame:
           129 log-magnitude spectrogram bins
         +  60 MFCC values (20 MFCCs plus first- and second-order deltas)
         +   4 sub-band energies  =  193 channels.
      5. Min-max normalise to [0, 1] using a single minimum and maximum computed
         jointly across the whole matrix of that recording.

    Returns: np.ndarray of shape (938, 193).
    """
    y, _ = librosa.load(wav_path, sr=sr)
    y = sosfiltfilt(_HPF_SOS, y)
    stft = librosa.stft(y, n_fft=256, hop_length=64, window='hann')
    spectrogram = librosa.amplitude_to_db(np.abs(stft))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=256, hop_length=64)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    mfccs = np.concatenate([mfcc, mfcc_delta, mfcc_delta2], axis=0)
    energy = np.array([
        np.sum(np.abs(stft[0:16, :]), axis=0),
        np.sum(np.abs(stft[16:32, :]), axis=0),
        np.sum(np.abs(stft[32:64, :]), axis=0),
        np.sum(np.abs(stft[0:128, :]), axis=0),
    ])
    features = np.concatenate([spectrogram, mfccs, energy], axis=0).T
    features = (features - features.min()) / (features.max() - features.min() + 1e-8)
    return features

def parse_label(label_path, sr=4000, hop_length=64, n_frames=938):
    """Convert a timestamped annotation file into frame-level binary labels.

    Each line of the label file has the form "<type> <start> <end>" with HH:MM:SS.mmm
    timestamps, which are converted to frame indices via sr / hop_length.
    Label groups: I = inhalation, E = exhalation,
    CAS = continuous adventitious sounds (wheeze / stridor / rhonchi, accepted in both
    single-letter and full-word spellings), DAS = discontinuous adventitious sounds
    (crackle, annotated as 'D').

    Returns: dict of four binary arrays, each of length n_frames.
    """
    labels = {'I': np.zeros(n_frames), 'E': np.zeros(n_frames),
              'CAS': np.zeros(n_frames), 'DAS': np.zeros(n_frames)}
    cas_types = {'W', 'S', 'R', 'Wheeze', 'Stridor', 'Rhonchi',
                 'wheeze', 'stridor', 'rhonchi'}
    def time_to_sec(t):
        h, m, s = t.split(':')
        return int(h)*3600 + int(m)*60 + float(s)
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3: continue
            sound_type = parts[0]
            start_frame = max(0, min(int(time_to_sec(parts[1]) * sr / hop_length), n_frames-1))
            end_frame = max(0, min(int(time_to_sec(parts[2]) * sr / hop_length), n_frames-1))
            if sound_type == 'I': labels['I'][start_frame:end_frame] = 1
            elif sound_type == 'E': labels['E'][start_frame:end_frame] = 1
            elif sound_type in cas_types: labels['CAS'][start_frame:end_frame] = 1
            elif sound_type == 'D': labels['DAS'][start_frame:end_frame] = 1
    return labels

class CNN_BiGRU(nn.Module):
    """CNN-BiGRU frame-level detector (Section III.C of the paper).

        Conv1d(193 -> 256, kernel 3) + ReLU + MaxPool1d(2) + Dropout(0.3)
     -> BiGRU (256 hidden units per direction, 512 concatenated)
     -> Linear(512 -> 1): one raw logit per output frame.

    MaxPool1d(2) halves the temporal resolution (938 -> 469 frames), which is why the
    labels are downsampled with [::2] in the Dataset classes. forward() returns logits;
    the sigmoid is applied separately at evaluation time.
    """

    def __init__(self, input_size=193, hidden_size=256, dropout=0.3):
        super(CNN_BiGRU, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, 256, kernel_size=3, padding=1), #Thanks to padding , the duration does not change.
            nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout))
        self.bigru = nn.GRU(256, hidden_size, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        x, _ = self.bigru(x)
        return self.fc(x).squeeze(-1)

class LungSoundDataset(Dataset):
    """Dataset over the preprocessed .npz files, used for the breath-phase baselines.

    Returns (features, label); the label is downsampled with [::2] so that it matches
    the model output resolution after MaxPool1d(2).
    """

    def __init__(self, data_path, task='inhalation', file_list=None):
        self.data_path = data_path
        self.task = task
        self.files = file_list if file_list is not None else \
            sorted(f for f in os.listdir(data_path) if f.endswith('.npz'))
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        data = np.load(os.path.join(self.data_path, self.files[idx]))
        features = torch.FloatTensor(data['features'])
        label = torch.FloatTensor(data[self.task])[::2]
        return features, label

def _date_key(filename):
    """Extract the recording-date identifier from a filename.

    Recordings made on the same date are grouped together so that closely related
    recordings are never split across train and validation (Section III.E, leakage
    prevention). Two filename conventions are present in HF_Lung_V1.
    """
    m = re.match(r'steth_(\d{8})_', filename)
    if m: return m.group(1)
    m = re.match(r'trunc_(\d{4}-\d{2}-\d{2})-', filename)
    if m: return m.group(1).replace('-', '')
    return filename

def split_train_val(data_path, val_frac=0.15, seed=42):
    """Date-grouped train/validation split over all recordings (breath-phase tasks).

    Date groups are shuffled with a fixed seed and assigned whole to the validation set
    until the target fraction is reached, so no recording date appears in both splits.
    """
    files = sorted(f for f in os.listdir(data_path) if f.endswith('.npz'))
    groups = defaultdict(list)
    for f in files:
        groups[_date_key(f)].append(f)
    group_keys = list(groups.keys())
    rng = np.random.RandomState(seed) #seed=42 is fixed, so the same shuffling order is produced every time the code runs.
    rng.shuffle(group_keys)
    target = int(len(files) * val_frac)
    train_files, val_files = [], []
    val_count = 0
    for k in group_keys:
        if val_count < target:
            val_files.extend(groups[k])
            val_count += len(groups[k])
        else:
            train_files.extend(groups[k])
    return sorted(train_files), sorted(val_files)

def compute_pos_weight(data_path, file_list, task):
    """Empirical negative-to-positive frame ratio, used as the raw BCE pos_weight."""
    total_pos, total = 0.0, 0
    for f in file_list:
        label = np.load(os.path.join(data_path, f))[task][::2]
        total_pos += label.sum()
        total += label.size
    total_neg = total - total_pos
    return float(total_neg / max(total_pos, 1.0))

def find_best_threshold(model, loader, device, thresholds=np.linspace(0.05, 0.95, 19)):
    """Grid-search the classification threshold over [0.05, 0.95] in steps of 0.05.

    This function is always called with the VALIDATION loader, never with the test set,
    so the decision threshold is selected without any test-set information.

    Returns: (best_threshold, best_F1).
    """
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for features, labels in loader:
            probs = torch.sigmoid(model(features.to(device))).cpu().numpy()
            all_probs.append(probs.flatten())
            all_labels.append(labels.numpy().astype(int).flatten())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        f1 = f1_score(all_labels, (all_probs > t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

print("Setup complete!")

Setup complete!


## 2. Core definitions: preprocessing, model and helpers

Defines the feature extraction pipeline, the label parser, the CNN-BiGRU architecture, the
date-grouped splitting helpers used to prevent leakage, and the validation-only threshold
search. This cell only defines functions, so it is cheap and should be run first in every
session.

In [ ]:
processed_path = '/content/drive/MyDrive/HF_Lung_V1-master/processed'

def process_dataset(data_path, save_path):
    """Preprocess every (wav, label) pair in data_path and cache the result as .npz.

    Files that already have a saved .npz are skipped, so this cell can be re-run safely
    without recomputing features. Each .npz stores the 938 x 193 feature matrix plus the
    four frame-level label arrays.
    """
    os.makedirs(save_path, exist_ok=True)
    files = os.listdir(data_path)
    wav_files = sorted([f for f in files if f.endswith('.wav')])
    pairs = [(os.path.join(data_path, w),
              os.path.join(data_path, w.replace('.wav', '_label.txt')))
             for w in wav_files if w.replace('.wav', '_label.txt') in files]
    print(f"Total pairs: {len(pairs)}")
    for wav_path, label_path in tqdm(pairs):
        save_file = os.path.join(save_path, os.path.basename(wav_path).replace('.wav', '.npz'))
        if os.path.exists(save_file): continue
        try:
            features = preprocess(wav_path)
            labels = parse_label(label_path)
            np.savez(save_file, features=features,
                     inhalation=labels['I'], exhalation=labels['E'],
                     cas=labels['CAS'], das=labels['DAS'])
        except Exception as e:
            print(f"Error: {e}")

os.makedirs('/content/drive/MyDrive/HF_Lung_V1-master/models', exist_ok=True)

print("Processing train...")
process_dataset('/content/drive/MyDrive/HF_Lung_V1-master/train/train',
                '/content/drive/MyDrive/HF_Lung_V1-master/processed/train')

print("Processing test...")
process_dataset('/content/drive/MyDrive/HF_Lung_V1-master/test/test',
                '/content/drive/MyDrive/HF_Lung_V1-master/processed/test')
print("Done! (already-processed files were skipped, not redone)")

Processing train...
Total pairs: 7809


100%|██████████| 7809/7809 [00:01<00:00, 5585.75it/s]


Processing test...
Total pairs: 1956


100%|██████████| 1956/1956 [00:00<00:00, 6124.14it/s]

Done! (already-processed files were skipped, not redone)


## 3. Reloading trained models (no retraining)

All models for inhalation, exhalation, CAS and DAS were trained in earlier sessions and
their weights are stored on Google Drive. The cells below reload those checkpoints rather
than retraining, so the reported numbers are reproduced exactly. Each reload still
recomputes the decision threshold on the validation split, never on the test set.

In [ ]:
def reload_baseline(task):
    """Reload a trained breath-phase model (inhalation / exhalation) from its checkpoint.

    No training happens here. The decision threshold is recomputed on the validation
    split so that it is reproduced exactly as it was originally selected.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt_path = f'/content/drive/MyDrive/HF_Lung_V1-master/models/best_{task}_model.pth'
    model = CNN_BiGRU().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    train_path = '/content/drive/MyDrive/HF_Lung_V1-master/processed/train'
    _, val_files = split_train_val(train_path, val_frac=0.15, seed=42)
    val_loader = DataLoader(LungSoundDataset(train_path, task=task, file_list=val_files),
                            batch_size=16, shuffle=False, num_workers=0)
    best_threshold, val_f1 = find_best_threshold(model, val_loader, device)
    print(f"[{task}] Reloaded. Threshold: {best_threshold:.2f} (val F1: {val_f1*100:.1f}%)")
    return model, best_threshold

inh_model, inh_threshold = reload_baseline('inhalation')
exh_model, exh_threshold = reload_baseline('exhalation')

KeyboardInterrupt: 

## 4. Event-positive filtering and the four loss configurations

CAS and DAS models are trained only on recordings that contain at least one event of the
target class, following Hsu et al. (2021). This already removes most of the class
imbalance before any loss-level intervention is applied.

In [ ]:
def split_filtered_train_val(data_path, task, val_frac=0.15, seed=42):
    """Date-grouped train/validation split restricted to EVENT-POSITIVE recordings.

    Only files containing at least one positive frame of `task` are kept, following the
    filtered-training protocol of Hsu et al. (2021). Grouping by recording date applies
    the same leakage-prevention rule as split_train_val.
    """
    files = sorted(f for f in os.listdir(data_path)
                    if f.endswith('.npz') and np.load(os.path.join(data_path, f))[task].sum() > 0)
    groups = defaultdict(list)
    for f in files:
        groups[_date_key(f)].append(f)
    group_keys = list(groups.keys())
    rng = np.random.RandomState(seed)
    rng.shuffle(group_keys)
    target = int(len(files) * val_frac)
    train_files, val_files = [], []
    val_count = 0
    for k in group_keys:
        if val_count < target:
            val_files.extend(groups[k])
            val_count += len(groups[k])
        else:
            train_files.extend(groups[k])
    return sorted(train_files), sorted(val_files)

In [ ]:
class _NpzTaskDataset(Dataset):
    """Minimal dataset over the preprocessed .npz files for a single task.

    Labels are downsampled with [::2] to match the model output resolution.
    """

    def __init__(self, path, files, task):
        self.path, self.files, self.task = path, files, task
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        data = np.load(os.path.join(self.path, self.files[idx]))
        return torch.FloatTensor(data['features']), torch.FloatTensor(data[self.task])[::2]

def _compute_pos_weight_from_files(path, files, task):
    """Empirical negative-to-positive frame ratio over the given (training) files."""
    total_pos, total = 0.0, 0
    for f in files:
        label = np.load(os.path.join(path, f))[task][::2]
        total_pos += label.sum()
        total += label.size
    return float((total - total_pos) / max(total_pos, 1.0))

def focal_loss_with_logits(logits, targets, alpha=0.25, gamma=2.0, reduction='mean'):
    """Lin et al. (2017), 'Focal Loss for Dense Object Detection'.
    alpha: weight for the POSITIVE class (data-derived here, not tuned on results).
    gamma: focusing parameter, standard default = 2.0 (not tuned)."""
    bce_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    p = torch.sigmoid(logits)
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    focal_weight = alpha_t * (1 - p_t) ** gamma
    loss = focal_weight * bce_loss
    return loss.mean() if reduction == 'mean' else loss.sum() if reduction == 'sum' else loss

def _compute_alpha_from_files(path, files, task):
    """alpha = 1 - pos_frac -- gives more weight to the minority (positive) class,
    same empirical, data-derived spirit as raw_pos_weight."""
    total_pos, total = 0.0, 0
    for f in files:
        label = np.load(os.path.join(path, f))[task][::2]
        total_pos += label.sum()
        total += label.size
    pos_frac = total_pos / total
    return float(1.0 - pos_frac)

def _ckpt_path(task, loss_mode, cap_value=None):
    """Checkpoint path for a task/loss configuration.

    Capped runs include the cap value in the filename so that different cap values can
    never silently overwrite each other's weights.
    """
    if loss_mode == 'capped':
        return f'/content/drive/MyDrive/HF_Lung_V1-master/models/best_{task}_capped_{cap_value}_v2.pth'
    return f'/content/drive/MyDrive/HF_Lung_V1-master/models/best_{task}_{loss_mode}_v2.pth'

def train_model_v2(task='cas', loss_mode='standard', cap_value=None, gamma=2.0, epochs=100, batch_size=16,
                    lr=0.0001, val_frac=0.15, seed=42, verbose_every=10, force_retrain=False):
    """Train (or reload) one model for a given task and loss configuration.

    loss_mode:
      'standard' -> BCEWithLogitsLoss with pos_weight = 1. This is the paper-consistent
                    baseline, since Hsu et al. (2021) apply no class weighting.
      'raw'      -> pos_weight = empirical negative-to-positive frame ratio.
      'capped'   -> pos_weight = cap_value. The assert below enforces
                    cap_value < raw_pos_weight, so a cap that would not actually moderate
                    the raw weight is rejected instead of silently producing a run that is
                    mathematically identical to 'raw'.
      'focal'    -> Focal Loss, with alpha derived from the training-set class balance and
                    gamma fixed at its standard default of 2.0 (neither is tuned).

    Leakage protection: the train/validation split is taken from the TRAINING directory
    only. Early stopping, learning-rate scheduling, checkpoint selection and threshold
    selection all use the validation set; the test partition is never read here.

    If a checkpoint already exists and force_retrain is False, the weights are reloaded
    instead of retrained (the threshold is still recomputed on the validation set).

    Returns: dict with the model, the validation-selected threshold, the validation F1,
    the weights actually used, and the file lists for the split.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_root = '/content/drive/MyDrive/HF_Lung_V1-master/processed/train'

    train_files, val_files = split_filtered_train_val(train_root, task, val_frac=val_frac, seed=seed)
    raw_pos_weight = _compute_pos_weight_from_files(train_root, train_files, task)

    alpha_value = None
    if loss_mode == 'standard':
        pos_weight_value = 1.0
    elif loss_mode == 'raw':
        pos_weight_value = raw_pos_weight
    elif loss_mode == 'capped':
        assert cap_value is not None and cap_value < raw_pos_weight, (
            f"cap_value ({cap_value}) must be set and < raw_pos_weight ({raw_pos_weight:.2f})"
        )
        pos_weight_value = cap_value
    elif loss_mode == 'focal':
        alpha_value = _compute_alpha_from_files(train_root, train_files, task)
        pos_weight_value = None
    else:
        raise ValueError(f"unknown loss_mode: {loss_mode}")

    ckpt_path = _ckpt_path(task, loss_mode, cap_value)
    val_dataset = _NpzTaskDataset(train_root, val_files, task)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    model = CNN_BiGRU().to(device)

    if not force_retrain and os.path.exists(ckpt_path):
        print(f"[{task}/{loss_mode}] Checkpoint found at {ckpt_path} -- RELOADING instead of retraining.")
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        best_threshold, val_f1 = find_best_threshold(model, val_loader, device)
        print(f"  Reloaded. Threshold from VALIDATION set: {best_threshold:.2f} (val F1: {val_f1*100:.1f}%)")
        return {
            'model': model, 'threshold': best_threshold, 'val_f1': val_f1,
            'raw_pos_weight': raw_pos_weight, 'pos_weight_used': pos_weight_value,
            'alpha_used': alpha_value, 'gamma_used': gamma if loss_mode == 'focal' else None,
            'train_files': train_files, 'val_files': val_files,
        }

    print(f"Train files: {len(train_files)}, Val files: {len(val_files)}")
    if loss_mode == 'focal':
        print(f"[{task}/focal] alpha={alpha_value:.3f} (data-derived), gamma={gamma} (standard default)")
    else:
        print(f"[{task}/{loss_mode}] raw_pos_weight={raw_pos_weight:.2f} -> using pos_weight={pos_weight_value:.2f}")

    train_dataset = _NpzTaskDataset(train_root, train_files, task)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if loss_mode == 'focal':
        criterion = lambda logits, labels: focal_loss_with_logits(logits, labels, alpha=alpha_value, gamma=gamma)
    else:
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=device))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.2)

    best_val_loss = float('inf')
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss += loss.item()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                val_loss += criterion(model(features), labels).item()
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
        if epoch % verbose_every == 0:
            print(f"  Epoch {epoch}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Best: {best_val_loss:.4f}")
        if no_improve >= 50:
            print(f"  Early stopping at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt_path))
    best_threshold, val_f1 = find_best_threshold(model, val_loader, device)
    print(f"[{task}/{loss_mode}] Best threshold from VALIDATION set: {best_threshold:.2f} (val F1: {val_f1*100:.1f}%)")

    return {
        'model': model, 'threshold': best_threshold, 'val_f1': val_f1,
        'raw_pos_weight': raw_pos_weight, 'pos_weight_used': pos_weight_value,
        'alpha_used': alpha_value, 'gamma_used': gamma if loss_mode == 'focal' else None,
        'train_files': train_files, 'val_files': val_files,
    }

## 5. Training / reloading the four loss configurations

Each call either trains the configuration or, if a checkpoint already exists, reloads it
and recomputes its validation-selected threshold.

In [ ]:
# Older CAS capped checkpoints were saved without the cap value in the filename.
# Copy that file to the cap-specific name so it is picked up by _ckpt_path(cap_value=2.5).
import shutil
old_path = '/content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_capped_v2.pth'
new_path = '/content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_capped_2.5_v2.pth'
if os.path.exists(old_path) and not os.path.exists(new_path):
    shutil.copy(old_path, new_path)
    print("CAS capped checkpoint (cap=2.5) copied under its new, cap-specific filename.")

# Train or reload the four CAS configurations. Cap = 2.5 was selected on the validation
# set and is below the CAS raw pos_weight, so the cap is genuinely active.
cas_results = {}
cas_results['standard'] = train_model_v2(task='cas', loss_mode='standard', epochs=100)
cas_results['raw'] = train_model_v2(task='cas', loss_mode='raw', epochs=100)
r = train_model_v2(task='cas', loss_mode='capped', cap_value=2.5, epochs=100)
r['cap_used'] = 2.5
cas_results['capped'] = r

[cas/standard] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_standard_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.20 (val F1: 61.7%)
[cas/raw] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_raw_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.55 (val F1: 60.1%)
[cas/capped] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_capped_2.5_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.45 (val F1: 60.2%)


In [ ]:
# Train or reload the DAS configurations. Cap = 1.5 is the validation-selected value for
# DAS; it differs from the CAS cap because the two tasks have different raw pos_weights.
das_results = {}
das_results['standard'] = train_model_v2(task='das', loss_mode='standard', epochs=100)
das_results['raw'] = train_model_v2(task='das', loss_mode='raw', epochs=100)
r = train_model_v2(task='das', loss_mode='capped', cap_value=1.5, epochs=100)
r['cap_used'] = 1.5
das_results['capped'] = r

[das/standard] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_das_standard_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.25 (val F1: 64.4%)
[das/raw] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_das_raw_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.50 (val F1: 62.9%)
[das/capped] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_das_capped_1.5_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.30 (val F1: 64.0%)


In [ ]:
# Focal Loss runs for both tasks. gamma is fixed at the standard literature default of
# 2.0 and alpha is derived from the training-set class balance inside train_model_v2.
cas_results['focal'] = train_model_v2(task='cas', loss_mode='focal', gamma=2.0, epochs=100)
das_results['focal'] = train_model_v2(task='das', loss_mode='focal', gamma=2.0, epochs=100)

[cas/focal] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_focal_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.50 (val F1: 57.3%)
[das/focal] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_das_focal_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.50 (val F1: 63.1%)


## 6. Evaluation functions (frame, segment and event level)

Event-level scoring converts frame sequences into discrete segments and matches predicted
segments to ground-truth segments using the temporal Jaccard Index, counting a match when
JI >= 0.5. This follows the evaluation methodology of Hsu et al. (2021), with the one
documented deviation that this study uses >= 0.5 where the original paper uses > 0.5.

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix

def frames_to_segments(binary_seq):
    """Convert a binary frame sequence into a list of (start, end) event segments."""
    segments = []
    in_seg = False
    start = 0
    for i, v in enumerate(binary_seq):
        if v == 1 and not in_seg:
            start = i
            in_seg = True
        elif v == 0 and in_seg:
            segments.append((start, i))
            in_seg = False
    if in_seg:
        segments.append((start, len(binary_seq)))
    return segments

def jaccard_index(seg_a, seg_b):
    """Temporal Jaccard Index (intersection over union) of two (start, end) segments."""
    s1, e1 = seg_a
    s2, e2 = seg_b
    inter = max(0, min(e1, e2) - max(s1, s2))
    union = max(e1, e2) - min(s1, s2)
    return inter / union if union > 0 else 0.0

def match_segments(pred_segments, true_segments, ji_threshold=0.5):
    """Greedily match predicted segments to ground-truth segments, one to one.

    A predicted segment counts as a true positive if its best Jaccard Index with an
    as-yet unmatched ground-truth segment is at least ji_threshold (0.5 throughout this
    study). Following Hsu et al. (2021), matching is used to obtain event-level counts.

    Returns: (TP, FP, FN) at the event level.
    """
    matched_true = set()
    tp = 0
    for p in pred_segments:
        best_ji, best_idx = 0.0, -1
        for idx, t in enumerate(true_segments):
            if idx in matched_true:
                continue
            ji = jaccard_index(p, t)
            if ji > best_ji:
                best_ji, best_idx = ji, idx
        if best_ji >= ji_threshold:
            tp += 1
            matched_true.add(best_idx)
    fp = len(pred_segments) - tp
    fn = len(true_segments) - tp
    return tp, fp, fn

def comprehensive_evaluate(model, task, threshold, data_path='/content/drive/MyDrive/HF_Lung_V1-master/processed/test',
                            file_list=None, ji_threshold=0.5, label_name=''):
    """Evaluate a trained model on the test partition at three levels of granularity.

    Frame-level:   accuracy, sensitivity, specificity, PPV, F1 and ROC-AUC over all frames.
    Segment-level: precision / recall / F1 computed from per-frame TP, FP and FN counts.
    Event-level:   precision / recall / F1 using Jaccard-Index matching (JI >= 0.5),
                   following the evaluation methodology of Hsu et al. (2021).

    `threshold` must be the value already selected on the validation set; this function
    performs no threshold search of its own.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    files = file_list if file_list is not None else \
        sorted(f for f in os.listdir(data_path) if f.endswith('.npz'))

    all_probs, all_labels = [], []
    tp_seg = fp_seg = fn_seg = 0
    tp_evt = fp_evt = fn_evt = 0

    with torch.no_grad():
        for fname in files:
            data = np.load(os.path.join(data_path, fname))
            features = torch.FloatTensor(data['features']).unsqueeze(0).to(device)
            label = data[task][::2]
            probs = torch.sigmoid(model(features)).cpu().numpy().flatten()
            preds = (probs > threshold).astype(int)

            all_probs.append(probs)
            all_labels.append(label.astype(int))

            pred_segments = frames_to_segments(preds)
            true_segments = frames_to_segments(label.astype(int))

            tp_seg += int(((preds == 1) & (label == 1)).sum())
            fp_seg += int(((preds == 1) & (label == 0)).sum())
            fn_seg += int(((preds == 0) & (label == 1)).sum())

            tp_e, fp_e, fn_e = match_segments(pred_segments, true_segments, ji_threshold)
            tp_evt += tp_e; fp_evt += fp_e; fn_evt += fn_e

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    all_preds = (all_probs > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    seg_precision = tp_seg / (tp_seg + fp_seg) if (tp_seg + fp_seg) > 0 else 0.0
    seg_recall = tp_seg / (tp_seg + fn_seg) if (tp_seg + fn_seg) > 0 else 0.0
    seg_f1 = 2 * seg_precision * seg_recall / (seg_precision + seg_recall) if (seg_precision + seg_recall) > 0 else 0.0

    evt_precision = tp_evt / (tp_evt + fp_evt) if (tp_evt + fp_evt) > 0 else 0.0
    evt_recall = tp_evt / (tp_evt + fn_evt) if (tp_evt + fn_evt) > 0 else 0.0
    evt_f1 = 2 * evt_precision * evt_recall / (evt_precision + evt_recall) if (evt_precision + evt_recall) > 0 else 0.0

    print(f"--- {label_name or task.upper()} ---")
    print(f"Frame-level  | Acc: {accuracy*100:.1f}% | Sens: {sensitivity*100:.1f}% | Spec: {specificity*100:.1f}% | PPV: {ppv*100:.1f}% | F1: {f1*100:.1f}% | AUC: {auc:.3f}")
    print(f"Segment-level| Precision: {seg_precision*100:.1f}% | Recall: {seg_recall*100:.1f}% | F1: {seg_f1*100:.1f}%")
    print(f"Event-level  | Precision: {evt_precision*100:.1f}% | Recall: {evt_recall*100:.1f}% | F1: {evt_f1*100:.1f}% (JI>={ji_threshold})")

    return {
        'accuracy': accuracy, 'sensitivity': sensitivity, 'specificity': specificity,
        'ppv': ppv, 'f1': f1, 'auc': auc,
        'segment_precision': seg_precision, 'segment_recall': seg_recall, 'segment_f1': seg_f1,
        'event_precision': evt_precision, 'event_recall': evt_recall, 'event_f1': evt_f1,
    }

## 7. Final held-out evaluation (Tables I and II)

This is the only point in the notebook where the predefined test partition is read. Every
threshold and checkpoint used here was selected earlier on the validation set. For CAS and
DAS, evaluation is restricted to event-positive test recordings, following the filtered
protocol of Hsu et al. (2021).

In [ ]:
# Final held-out evaluation. This is the ONLY place where the test partition is read.
# For CAS and DAS the evaluation is restricted to event-positive test recordings,
# following the filtered protocol of Hsu et al. (2021).
results = {}
results['inhalation_baseline'] = comprehensive_evaluate(inh_model, 'inhalation', inh_threshold, label_name='Inhalation (baseline)')
results['exhalation_baseline'] = comprehensive_evaluate(exh_model, 'exhalation', exh_threshold, label_name='Exhalation (baseline)')

for task, results_dict in [('cas', cas_results), ('das', das_results)]:
    test_root = '/content/drive/MyDrive/HF_Lung_V1-master/processed/test'
    test_files = [f for f in os.listdir(test_root)
                  if f.endswith('.npz') and np.load(os.path.join(test_root, f))[task].sum() > 0]
    for mode in ['standard', 'raw', 'capped', 'focal']:
        r = results_dict[mode]
        label = f"{task.upper()} ({mode}"
        if mode == 'capped':
            label += f", cap={r['cap_used']}"
        label += ")"
        results[f'{task}_{mode}'] = comprehensive_evaluate(
            r['model'], task, r['threshold'], file_list=test_files, label_name=label
        )

import pandas as pd
df = pd.DataFrame(results).T
df

--- Inhalation (baseline) ---
Frame-level  | Acc: 89.3% | Sens: 78.5% | Spec: 92.3% | PPV: 73.7% | F1: 76.0% | AUC: 0.941
Segment-level| Precision: 73.7% | Recall: 78.5% | F1: 76.0%
Event-level  | Precision: 70.8% | Recall: 85.4% | F1: 77.4% (JI>=0.5)
--- Exhalation (baseline) ---
Frame-level  | Acc: 88.7% | Sens: 50.3% | Spec: 92.5% | PPV: 40.4% | F1: 44.8% | AUC: 0.849
Segment-level| Precision: 40.4% | Recall: 50.3% | F1: 44.8%
Event-level  | Precision: 31.5% | Recall: 51.2% | F1: 39.0% (JI>=0.5)
--- CAS (standard) ---
Frame-level  | Acc: 76.3% | Sens: 64.0% | Spec: 79.1% | PPV: 41.3% | F1: 50.2% | AUC: 0.802
Segment-level| Precision: 41.3% | Recall: 64.0% | F1: 50.2%
Event-level  | Precision: 34.8% | Recall: 46.7% | F1: 39.9% (JI>=0.5)
--- CAS (raw) ---
Frame-level  | Acc: 76.2% | Sens: 66.5% | Spec: 78.5% | PPV: 41.6% | F1: 51.2% | AUC: 0.808
Segment-level| Precision: 41.6% | Recall: 66.5% | F1: 51.2%
Event-level  | Precision: 33.7% | Recall: 49.5% | F1: 40.1% (JI>=0.5)
--- CAS (ca

,accuracy,sensitivity,specificity,ppv,f1,auc,segment_precision,segment_recall,segment_f1,event_precision,event_recall,event_f1
inhalation_baseline,0.892857,0.785335,0.922577,0.737096,0.760451,0.941476,0.737096,0.785335,0.760451,0.708389,0.854170,0.774479
exhalation_baseline,0.886600,0.503256,0.925241,0.404251,0.448353,0.848827,0.404251,0.503256,0.448353,0.315400,0.512195,0.390400
cas_standard,0.762565,0.640156,0.790754,0.413328,0.502322,0.802465,0.413328,0.640156,0.502322,0.348125,0.467282,0.398997
cas_raw,0.762259,0.665403,0.784563,0.415639,0.511668,0.808315,0.415639,0.665403,0.511668,0.336947,0.495386,0.401087
cas_capped,0.763904,0.571035,0.808319,0.406896,0.475191,0.799072,0.406896,0.571035,0.475191,0.328212,0.394295,0.358232
cas_focal,0.703570,0.726666,0.698251,0.356737,0.478545,0.773416,0.356737,0.726666,0.478545,0.301366,0.462668,0.364990
das_standard,0.785193,0.845972,0.759582,0.597216,0.700155,0.874820,0.597216,0.845972,0.700155,0.521371,0.653460,0.579990
das_raw,0.784434,0.829281,0.765536,0.598454,0.695208,0.872030,0.598454,0.829281,0.695208,0.507885,0.611049,0.554711
das_capped,0.772846,0.874370,0.730066,0.577153,0.695332,0.872730,0.577153,0.874370,0.695332,0.522459,0.616629,0.565651
das_focal,0.769445,0.847477,0.736564,0.575474,0.685479,0.868650,0.575474,0.847477,0.685479,0.477151,0.594308,0.529324


## 8. Multi-seed robustness check (Table III)

Purpose: determine whether the small single-run differences are real or simply training
noise, specifically `CAS raw (51.2%) vs standard (50.2%)` and `DAS capped sensitivity
(87.4%) vs standard (84.6%)`.

Important: the train/validation **split** is always fixed with `split_seed=42`, so every
seeded run is evaluated on the identical validation set. Only the model initialisation and
the batch ordering vary (`init_seed`). What is measured is therefore pure training
stochasticity, not a different validation split.

In [ ]:
import random

def _ckpt_path_seeded(task, loss_mode, init_seed, cap_value=None):
    """Per-seed checkpoint path, stored in a separate seeds/ folder so that runs with
    different initialisation seeds never overwrite one another."""
    os.makedirs('/content/drive/MyDrive/HF_Lung_V1-master/models/seeds', exist_ok=True)
    if loss_mode == 'capped':
        return f'/content/drive/MyDrive/HF_Lung_V1-master/models/seeds/best_{task}_capped_{cap_value}_seed{init_seed}.pth'
    return f'/content/drive/MyDrive/HF_Lung_V1-master/models/seeds/best_{task}_{loss_mode}_seed{init_seed}.pth'

def train_model_seeded(task, loss_mode, init_seed, cap_value=None, gamma=2.0, epochs=100, batch_size=16,
                        lr=0.0001, val_frac=0.15, split_seed=42, verbose_every=20, force_retrain=False):
    """Same protocol as train_model_v2, but:
      - the train/val SPLIT always uses split_seed=42 (fixed across every seed run here, so all
        runs are evaluated on the identical validation set -- only init/shuffling vary);
      - model init + DataLoader shuffling are seeded with init_seed;
      - checkpoints are stored per-seed (seeds/ subfolder) so they never overwrite each other."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_root = '/content/drive/MyDrive/HF_Lung_V1-master/processed/train'

    train_files, val_files = split_filtered_train_val(train_root, task, val_frac=val_frac, seed=split_seed)
    raw_pos_weight = _compute_pos_weight_from_files(train_root, train_files, task)

    alpha_value = None
    if loss_mode == 'standard':
        pos_weight_value = 1.0
    elif loss_mode == 'raw':
        pos_weight_value = raw_pos_weight
    elif loss_mode == 'capped':
        assert cap_value is not None and cap_value < raw_pos_weight
        pos_weight_value = cap_value
    elif loss_mode == 'focal':
        alpha_value = _compute_alpha_from_files(train_root, train_files, task)
        pos_weight_value = None
    else:
        raise ValueError(f"unknown loss_mode: {loss_mode}")

    ckpt_path = _ckpt_path_seeded(task, loss_mode, init_seed, cap_value)

    random.seed(init_seed)
    np.random.seed(init_seed)
    torch.manual_seed(init_seed)
    torch.cuda.manual_seed_all(init_seed)

    val_dataset = _NpzTaskDataset(train_root, val_files, task)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    model = CNN_BiGRU().to(device)

    if not force_retrain and os.path.exists(ckpt_path):
        print(f"[{task}/{loss_mode}/seed={init_seed}] Checkpoint found -- RELOADING.")
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        best_threshold, val_f1 = find_best_threshold(model, val_loader, device)
        print(f"  Reloaded. val F1: {val_f1*100:.1f}%")
        return {'val_f1': val_f1, 'threshold': best_threshold, 'model': model}

    train_dataset = _NpzTaskDataset(train_root, train_files, task)
    g = torch.Generator()
    g.manual_seed(init_seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, generator=g)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if loss_mode == 'focal':
        criterion = lambda logits, labels: focal_loss_with_logits(logits, labels, alpha=alpha_value, gamma=gamma)
    else:
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=device))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.2)

    best_val_loss = float('inf')
    no_improve = 0
    print(f"[{task}/{loss_mode}/seed={init_seed}] training...")
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                val_loss += criterion(model(features), labels).item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
        if epoch % verbose_every == 0:
            print(f"  Epoch {epoch}/{epochs} | Val: {val_loss:.4f} | Best: {best_val_loss:.4f}")
        if no_improve >= 50:
            print(f"  Early stopping at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt_path))
    best_threshold, val_f1 = find_best_threshold(model, val_loader, device)
    print(f"[{task}/{loss_mode}/seed={init_seed}] val F1: {val_f1*100:.1f}%")
    return {'val_f1': val_f1, 'threshold': best_threshold, 'model': model}

In [ ]:
# Priority 1: re-run the two most important single-run findings across three seeds.
#   CAS: standard vs raw     (51.2 vs 50.2 -- a real difference, or single-run noise?)
#   DAS: standard vs capped  (equal F1, but capped showed higher sensitivity -- does it replicate?)
SEEDS = [42, 123, 2026]

priority1 = [
    ('cas', 'standard', None),
    ('cas', 'raw', None),
    ('das', 'standard', None),
    ('das', 'capped', 1.5),
]

multiseed_results = {}
test_root = '/content/drive/MyDrive/HF_Lung_V1-master/processed/test'

for task, mode, cap in priority1:
    key = f"{task}_{mode}" if cap is None else f"{task}_{mode}_cap{cap}"
    multiseed_results[key] = {'val_f1': [], 'test_f1': []}
    test_files = [f for f in os.listdir(test_root)
                  if f.endswith('.npz') and np.load(os.path.join(test_root, f))[task].sum() > 0]
    for s in SEEDS:
        r = train_model_seeded(task=task, loss_mode=mode, cap_value=cap, init_seed=s, epochs=100)
        test_metrics = comprehensive_evaluate(r['model'], task, r['threshold'], file_list=test_files,
                                               label_name=f"{key} seed={s}")
        multiseed_results[key]['val_f1'].append(r['val_f1'])
        multiseed_results[key]['test_f1'].append(test_metrics['f1'])

    v = np.array(multiseed_results[key]['val_f1']) * 100
    t = np.array(multiseed_results[key]['test_f1']) * 100
    print(f"\n{key} | val F1 per seed: {[f'{x:.1f}' for x in v]} -> mean={v.mean():.1f} sd={v.std():.2f}")
    print(f"{key} | test F1 per seed: {[f'{x:.1f}' for x in t]} -> mean={t.mean():.1f} sd={t.std():.2f}\n")

[cas/standard/seed=42] Checkpoint found -- RELOADING.
  Reloaded. val F1: 58.9%
--- cas_standard seed=42 ---
Frame-level  | Acc: 75.8% | Sens: 55.1% | Spec: 80.6% | PPV: 39.6% | F1: 46.1% | AUC: 0.784
Segment-level| Precision: 39.6% | Recall: 55.1% | F1: 46.1%
Event-level  | Precision: 30.3% | Recall: 37.8% | F1: 33.7% (JI>=0.5)
[cas/standard/seed=123] training...
  Epoch 0/100 | Val: 0.6535 | Best: 0.6535
  Epoch 20/100 | Val: 0.5307 | Best: 0.5228
  Epoch 40/100 | Val: 0.5292 | Best: 0.5076
  Epoch 60/100 | Val: 0.5393 | Best: 0.5076
  Early stopping at epoch 75
[cas/standard/seed=123] val F1: 59.2%
--- cas_standard seed=123 ---
Frame-level  | Acc: 76.5% | Sens: 60.4% | Spec: 80.3% | PPV: 41.3% | F1: 49.1% | AUC: 0.798
Segment-level| Precision: 41.3% | Recall: 60.4% | F1: 49.1%
Event-level  | Precision: 35.0% | Recall: 44.5% | F1: 39.2% (JI>=0.5)
[cas/standard/seed=2026] training...
  Epoch 0/100 | Val: 0.6380 | Best: 0.6380
  Epoch 20/100 | Val: 0.5670 | Best: 0.5290
  Epoch 40/100 

KeyboardInterrupt: 

In [ ]:
# Complete the DAS runs: force-retrain seed=123 (its earlier run was interrupted), then
# run seed=2026 and all three das_capped seeds normally (reloaded if already trained).
das_seed_configs = [
    ('das', 'standard', None, 123, True),   # interrupted earlier -- force a full retrain
    ('das', 'standard', None, 2026, False),
    ('das', 'capped', 1.5, 42, False),
    ('das', 'capped', 1.5, 123, False),
    ('das', 'capped', 1.5, 2026, False),
]

if 'multiseed_results' not in dir():
    multiseed_results = {}

test_root = '/content/drive/MyDrive/HF_Lung_V1-master/processed/test'

# das_standard/seed=42 had already been trained to completion (test F1: 69.4%). Reload it
# from its checkpoint and add it to the results list so that the run is not lost.
if 'das_standard' not in multiseed_results:
    multiseed_results['das_standard'] = {'val_f1': [], 'test_f1': []}
    test_files_das = [f for f in os.listdir(test_root)
                       if f.endswith('.npz') and np.load(os.path.join(test_root, f))['das'].sum() > 0]
    r42 = train_model_seeded(task='das', loss_mode='standard', init_seed=42, epochs=100)
    m42 = comprehensive_evaluate(r42['model'], 'das', r42['threshold'], file_list=test_files_das,
                                  label_name="das_standard seed=42 (reload)")
    multiseed_results['das_standard']['val_f1'].append(r42['val_f1'])
    multiseed_results['das_standard']['test_f1'].append(m42['f1'])

for task, mode, cap, seed, force in das_seed_configs:
    key = f"{task}_{mode}" if cap is None else f"{task}_{mode}_cap{cap}"
    if key not in multiseed_results:
        multiseed_results[key] = {'val_f1': [], 'test_f1': []}
    test_files = [f for f in os.listdir(test_root)
                  if f.endswith('.npz') and np.load(os.path.join(test_root, f))[task].sum() > 0]
    r = train_model_seeded(task=task, loss_mode=mode, cap_value=cap, init_seed=seed,
                            epochs=100, force_retrain=force)
    m = comprehensive_evaluate(r['model'], task, r['threshold'], file_list=test_files,
                                label_name=f"{key} seed={seed}")
    multiseed_results[key]['val_f1'].append(r['val_f1'])
    multiseed_results[key]['test_f1'].append(m['f1'])

for key in ['das_standard', 'das_capped_cap1.5']:
    if key in multiseed_results:
        v = np.array(multiseed_results[key]['val_f1']) * 100
        t = np.array(multiseed_results[key]['test_f1']) * 100
        print(f"\n{key} | val F1 per seed: {[f'{x:.1f}' for x in v]} -> mean={v.mean():.1f} sd={v.std():.2f}")
        print(f"{key} | test F1 per seed: {[f'{x:.1f}' for x in t]} -> mean={t.mean():.1f} sd={t.std():.2f}\n")

[das/standard/seed=123] training...
  Epoch 0/100 | Val: 0.5316 | Best: 0.5316
  Epoch 20/100 | Val: 0.3863 | Best: 0.3863
  Epoch 40/100 | Val: 0.3709 | Best: 0.3666
  Epoch 60/100 | Val: 0.3702 | Best: 0.3653
  Epoch 80/100 | Val: 0.3612 | Best: 0.3611
[das/standard/seed=123] val F1: 64.5%
--- das_standard seed=123 ---
Frame-level  | Acc: 80.1% | Sens: 81.6% | Spec: 79.4% | PPV: 62.6% | F1: 70.8% | AUC: 0.874
Segment-level| Precision: 62.6% | Recall: 81.6% | F1: 70.8%
Event-level  | Precision: 55.7% | Recall: 68.4% | F1: 61.4% (JI>=0.5)
[das/standard/seed=2026] training...
  Epoch 0/100 | Val: 0.5251 | Best: 0.5251
  Epoch 20/100 | Val: 0.3988 | Best: 0.3849
  Epoch 40/100 | Val: 0.3774 | Best: 0.3636
  Epoch 60/100 | Val: 0.3639 | Best: 0.3596
  Epoch 80/100 | Val: 0.3601 | Best: 0.3573
[das/standard/seed=2026] val F1: 63.6%
--- das_standard seed=2026 ---
Frame-level  | Acc: 77.6% | Sens: 86.2% | Spec: 73.9% | PPV: 58.2% | F1: 69.5% | AUC: 0.875
Segment-level| Precision: 58.2% | Rec

In [ ]:
# Priority 2 (time permitting): check whether the high Focal Loss CAS sensitivity
# observed in the single run (72.7%) replicates across seeds.
for task in ['cas', 'das']:
    key = f"{task}_focal"
    multiseed_results[key] = {'val_f1': [], 'test_f1': []}
    test_files = [f for f in os.listdir(test_root)
                  if f.endswith('.npz') and np.load(os.path.join(test_root, f))[task].sum() > 0]
    for s in SEEDS:
        r = train_model_seeded(task=task, loss_mode='focal', init_seed=s, epochs=100)
        test_metrics = comprehensive_evaluate(r['model'], task, r['threshold'], file_list=test_files,
                                               label_name=f"{key} seed={s}")
        multiseed_results[key]['val_f1'].append(r['val_f1'])
        multiseed_results[key]['test_f1'].append(test_metrics['f1'])

    v = np.array(multiseed_results[key]['val_f1']) * 100
    t = np.array(multiseed_results[key]['test_f1']) * 100
    print(f"\n{key} | val F1 per seed: {[f'{x:.1f}' for x in v]} -> mean={v.mean():.1f} sd={v.std():.2f}")
    print(f"{key} | test F1 per seed: {[f'{x:.1f}' for x in t]} -> mean={t.mean():.1f} sd={t.std():.2f}\n")

In [ ]:
# Summarise every multi-seed configuration: mean and standard deviation of validation and
# test F1 across seeds, plus the individual per-seed test scores (Table III of the paper).
import pandas as pd
rows = []
for key, d in multiseed_results.items():
    v, t = np.array(d['val_f1'])*100, np.array(d['test_f1'])*100
    rows.append({'config': key, 'val_mean': v.mean(), 'val_sd': v.std(),
                 'test_mean': t.mean(), 'test_sd': t.std(), 'test_per_seed': [round(x,1) for x in t]})
pd.DataFrame(rows)

## 8b. Follow-up runs: detecting and repairing an incomplete checkpoint

The sanity check in the DAS raw-weighted cell below flagged `seed=2026` as an outlier
(validation F1 of 36.2% against a median of 55.6%), which indicated a checkpoint written
during an interrupted run rather than a genuine result. That seed is retrained with
`force_retrain=True`, re-evaluated, and the corrected value is used in the final summary.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
r = train_model_seeded(task='das', loss_mode='raw', init_seed=2026, epochs=100, verbose_every=1)

[das/raw/seed=2026] Checkpoint found -- RELOADING.
  Reloaded. val F1: 36.2%


In [ ]:
# Three-seed run for the DAS raw-weighted configuration, completing the symmetric
# comparison against DAS standard BCE.
das_raw_results = {'val_f1': [], 'test_f1': []}
test_root = '/content/drive/MyDrive/HF_Lung_V1-master/processed/test'
test_files_das = [f for f in os.listdir(test_root)
                   if f.endswith('.npz') and np.load(os.path.join(test_root, f))['das'].sum() > 0]

for seed in [42, 123, 2026]:
    r = train_model_seeded(task='das', loss_mode='raw', init_seed=seed, epochs=100)
    m = comprehensive_evaluate(r['model'], 'das', r['threshold'], file_list=test_files_das,
                                label_name=f"das_raw seed={seed}")
    das_raw_results['val_f1'].append(r['val_f1'])
    das_raw_results['test_f1'].append(m['f1'])

v = np.array(das_raw_results['val_f1']) * 100
t = np.array(das_raw_results['test_f1']) * 100

# Sanity check: warn if one seed deviates strongly from the others, which usually means
# its checkpoint was written during an interrupted run and needs to be retrained.
median_v = np.median(v)
for seed, val in zip([42, 123, 2026], v):
    if abs(val - median_v) > 10:  # a gap of more than 10 points is suspicious
        print(f"WARNING: seed={seed} val F1 ({val:.1f}%) differs strongly from the others "
              f"({median_v:.1f}% median) -- the checkpoint is probably incomplete; re-run this "
              f"seed on its own with force_retrain=True.")

print(f"\ndas_raw | val F1 per seed: {[f'{x:.1f}' for x in v]} -> mean={v.mean():.1f} sd={v.std():.2f}")
print(f"das_raw | test F1 per seed: {[f'{x:.1f}' for x in t]} -> mean={t.mean():.1f} sd={t.std():.2f}\n")

[das/raw/seed=42] Checkpoint found -- RELOADING.
  Reloaded. val F1: 56.7%
--- das_raw seed=42 ---
Frame-level  | Acc: 76.4% | Sens: 80.4% | Spec: 74.7% | PPV: 57.2% | F1: 66.8% | AUC: 0.850
Segment-level| Precision: 57.2% | Recall: 80.4% | F1: 66.8%
Event-level  | Precision: 54.2% | Recall: 63.3% | F1: 58.4% (JI>=0.5)
[das/raw/seed=123] Checkpoint found -- RELOADING.
  Reloaded. val F1: 55.6%
--- das_raw seed=123 ---
Frame-level  | Acc: 69.9% | Sens: 89.6% | Spec: 61.7% | PPV: 49.6% | F1: 63.9% | AUC: 0.844
Segment-level| Precision: 49.6% | Recall: 89.6% | F1: 63.9%
Event-level  | Precision: 43.4% | Recall: 51.2% | F1: 46.9% (JI>=0.5)
[das/raw/seed=2026] Checkpoint found -- RELOADING.
  Reloaded. val F1: 36.2%
--- das_raw seed=2026 ---
Frame-level  | Acc: 30.4% | Sens: 99.6% | Spec: 1.2% | PPV: 29.8% | F1: 45.9% | AUC: 0.786
Segment-level| Precision: 29.8% | Recall: 99.6% | F1: 45.9%
Event-level  | Precision: 0.0% | Recall: 0.0% | F1: 0.0% (JI>=0.5)

das_raw | val F1 per seed: ['56.7'

In [ ]:
# The sanity check above flagged seed=2026 as an incomplete checkpoint, so it is retrained
# from scratch with force_retrain=True.
r2026 = train_model_seeded(task='das', loss_mode='raw', init_seed=2026, epochs=100,
                            verbose_every=1, force_retrain=True)

[das/raw/seed=2026] training...
  Epoch 0/100 | Val: 0.9424 | Best: 0.9424
  Epoch 1/100 | Val: 0.9336 | Best: 0.9336
  Epoch 2/100 | Val: 0.9409 | Best: 0.9336
  Epoch 3/100 | Val: 0.9182 | Best: 0.9182
  Epoch 4/100 | Val: 0.8072 | Best: 0.8072
  Epoch 5/100 | Val: 0.8325 | Best: 0.8072
  Epoch 6/100 | Val: 0.8464 | Best: 0.8072
  Epoch 7/100 | Val: 0.7409 | Best: 0.7409
  Epoch 8/100 | Val: 0.7300 | Best: 0.7300
  Epoch 9/100 | Val: 0.7567 | Best: 0.7300
  Epoch 10/100 | Val: 0.6911 | Best: 0.6911
  Epoch 11/100 | Val: 0.6919 | Best: 0.6911
  Epoch 12/100 | Val: 0.6745 | Best: 0.6745
  Epoch 13/100 | Val: 0.6595 | Best: 0.6595
  Epoch 14/100 | Val: 0.6618 | Best: 0.6595
  Epoch 15/100 | Val: 0.6530 | Best: 0.6530
  Epoch 16/100 | Val: 0.6518 | Best: 0.6518
  Epoch 17/100 | Val: 0.6501 | Best: 0.6501
  Epoch 18/100 | Val: 0.6701 | Best: 0.6501
  Epoch 19/100 | Val: 0.6650 | Best: 0.6501
  Epoch 20/100 | Val: 0.7438 | Best: 0.6501
  Epoch 21/100 | Val: 0.6445 | Best: 0.6445
  Epoch 22

In [ ]:
# Re-evaluate the retrained seed=2026 model on the event-positive DAS test recordings.
import os, numpy as np

test_root = '/content/drive/MyDrive/HF_Lung_V1-master/processed/test'
test_files_das = [f for f in os.listdir(test_root)
                   if f.endswith('.npz') and np.load(os.path.join(test_root, f))['das'].sum() > 0]

m2026 = comprehensive_evaluate(r2026['model'], 'das', r2026['threshold'],
                                file_list=test_files_das, label_name="das_raw seed=2026 (fixed)")

--- das_raw seed=2026 (fixed) ---
Frame-level  | Acc: 77.3% | Sens: 83.1% | Spec: 74.8% | PPV: 58.1% | F1: 68.4% | AUC: 0.864
Segment-level| Precision: 58.1% | Recall: 83.1% | F1: 68.4%
Event-level  | Precision: 48.1% | Recall: 58.1% | F1: 52.6% (JI>=0.5)


In [ ]:
# Final DAS raw-weighted summary: the two valid seeds from the earlier run (42 and 123)
# combined with the retrained seed=2026 result computed immediately above.
das_raw_results = {
    'val_f1':  [0.567, 0.556, 0.628],
    'test_f1': [0.668, 0.639, m2026['f1']],
}

v = np.array(das_raw_results['val_f1']) * 100
t = np.array(das_raw_results['test_f1']) * 100
print(f"das_raw | val F1 per seed: {[f'{x:.1f}' for x in v]} -> mean={v.mean():.1f} sd={v.std():.2f}")
print(f"das_raw | test F1 per seed: {[f'{x:.1f}' for x in t]} -> mean={t.mean():.1f} sd={t.std():.2f}")

das_raw | val F1 per seed: ['56.7', '55.6', '62.8'] -> mean=58.4 sd=3.17
das_raw | test F1 per seed: ['66.8', '63.9', '68.4'] -> mean=66.4 sd=1.86


## 9. Duration-stratified error analysis (Table IV)

Reload the single-run standard and raw-weighted checkpoints, then examine which kinds of
events are affected by raw class weighting. Ground-truth events are split into short,
medium and long bins using the 33rd and 67th percentiles of event duration within each
task's test set, and recall is computed separately in each bin, alongside the number of
false-positive events per recording.

In [ ]:
import time


def _reload_with_progress(task, loss_mode, cap_value=None):
    """Reload a single-run checkpoint, printing timestamps so a slow first call can be
    distinguished from a hung one. The first call is slow because
    split_filtered_train_val rescans the whole training directory over Google Drive;
    later calls hit the warmed filesystem cache."""
    t0 = time.time()
    print(f">>> {task}/{loss_mode} started ({time.strftime('%H:%M:%S')})...")
    r = train_model_v2(task=task, loss_mode=loss_mode, cap_value=cap_value, epochs=100)
    print(f"<<< {task}/{loss_mode} finished in {time.time() - t0:.1f} s. "
          f"threshold={r['threshold']:.2f}, val_f1={r['val_f1']*100:.1f}%\n")
    return r

cas_results = {}
cas_results['standard'] = _reload_with_progress('cas', 'standard')
cas_results['raw']      = _reload_with_progress('cas', 'raw')
das_results = {}
das_results['standard'] = _reload_with_progress('das', 'standard')
das_results['raw']      = _reload_with_progress('das', 'raw')

print("All done.")

>>> cas/standard started (14:59:43)...
[cas/standard] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_standard_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.20 (val F1: 61.7%)
<<< cas/standard finished in 655.8 s. threshold=0.20, val_f1=61.7%

>>> cas/raw started (15:10:38)...
[cas/raw] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_cas_raw_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.55 (val F1: 60.1%)
<<< cas/raw finished in 23.8 s. threshold=0.55, val_f1=60.1%

>>> das/standard started (15:11:02)...
[das/standard] Checkpoint found at /content/drive/MyDrive/HF_Lung_V1-master/models/best_das_standard_v2.pth -- RELOADING instead of retraining.
  Reloaded. Threshold from VALIDATION set: 0.25 (val F1: 64.4%)
<<< das/standard finished in 40.3 s. threshold=0.25, val_f1=64.4%

>>> das/raw started (15:11:43)...
[das/raw] Checkpoint found at /content/driv

In [ ]:
import os
import numpy as np
import pandas as pd
import torch

TEST_ROOT = '/content/drive/MyDrive/HF_Lung_V1-master/processed/test'
FRAME_SECONDS = (64 * 2) / 4000  # = 0.032 s per output frame (hop 64, halved by MaxPool1d(2))


def match_segments_with_indices(pred_segments, true_segments, ji_threshold=0.5):
    """Same JI >= threshold matching rule as match_segments(), but it also returns WHICH
    ground-truth event each prediction matched. Those indices are required in order to
    compute recall separately within each event-duration bin."""
    matched_true = set()
    matches = []
    for pred_seg in pred_segments:
        best_ji, best_true_idx = 0.0, -1
        for true_idx, true_seg in enumerate(true_segments):
            if true_idx in matched_true:
                continue
            ji = jaccard_index(pred_seg, true_seg)
            if ji > best_ji:
                best_ji, best_true_idx = ji, true_idx
        if best_true_idx >= 0 and best_ji >= ji_threshold:
            matched_true.add(best_true_idx)
            matches.append((best_true_idx, best_ji))
    return matches


def collect_event_errors(model, task, threshold, test_files, ji_threshold=0.5):
    """For one model, record the duration of every ground-truth event, whether that event
    was detected, and the number of false-positive events per recording."""
    device = next(model.parameters()).device
    model.eval()
    gt_durations, gt_matched = [], []
    total_fp_events = 0
    with torch.no_grad():
        for fname in test_files:
            data = np.load(os.path.join(TEST_ROOT, fname))
            features = torch.FloatTensor(data['features']).unsqueeze(0).to(device)
            labels = data[task][::2].astype(int)
            probs = torch.sigmoid(model(features)).cpu().numpy().ravel()
            preds = (probs > threshold).astype(int)

            true_segments = frames_to_segments(labels)
            pred_segments = frames_to_segments(preds)
            matches = match_segments_with_indices(pred_segments, true_segments, ji_threshold)
            matched_true_indices = {m[0] for m in matches}
            total_fp_events += len(pred_segments) - len(matches)

            for true_idx, (start, end) in enumerate(true_segments):
                gt_durations.append((end - start) * FRAME_SECONDS)
                gt_matched.append(int(true_idx in matched_true_indices))

    return {
        'durations': np.asarray(gt_durations),
        'matched': np.asarray(gt_matched),
        'fp_per_recording': total_fp_events / len(test_files),
    }


def duration_recall_table(task, configs, test_files):
    """Build the duration-stratified recall table (Table IV of the paper).

    Ground-truth events are binned into short / medium / long using the 33rd and 67th
    percentiles of event duration within that task's test set, and recall is computed
    separately inside each bin. The bin edges are model-independent, since all
    configurations are evaluated on the same ground-truth events.

    configs maps a display name to a train_model_v2 result dict.
    """
    collected = {name: collect_event_errors(r['model'], task, r['threshold'], test_files)
                 for name, r in configs.items()}
    durations = next(iter(collected.values()))['durations']
    q33, q67 = np.quantile(durations, [1 / 3, 2 / 3])

    rows = []
    for name, result in collected.items():
        d, matched = result['durations'], result['matched']
        masks = {
            f'Short (<{q33:.2f}s)': d < q33,
            f'Medium ({q33:.2f}-{q67:.2f}s)': (d >= q33) & (d < q67),
            f'Long (>={q67:.2f}s)': d >= q67,
        }
        row = {'Task': task.upper(), 'Model': name,
               'FP events/recording': round(result['fp_per_recording'], 2)}
        for label, mask in masks.items():
            row[f'{label} recall (%)'] = round(100 * matched[mask].mean(), 1) if mask.any() else float('nan')
            row[f'{label} n'] = int(mask.sum())
        rows.append(row)
    return pd.DataFrame(rows), (q33, q67)


# Restrict each task to its event-positive test recordings, exactly as in Table I.
test_files_cas = [f for f in os.listdir(TEST_ROOT)
                   if f.endswith('.npz') and np.load(os.path.join(TEST_ROOT, f))['cas'].sum() > 0]
test_files_das = [f for f in os.listdir(TEST_ROOT)
                   if f.endswith('.npz') and np.load(os.path.join(TEST_ROOT, f))['das'].sum() > 0]

cas_table, cas_bins = duration_recall_table(
    task='cas',
    configs={'Standard BCE': cas_results['standard'], 'Raw weighted BCE': cas_results['raw']},
    test_files=test_files_cas,
)
das_table, das_bins = duration_recall_table(
    task='das',
    configs={'Standard BCE': das_results['standard'], 'Raw weighted BCE': das_results['raw']},
    test_files=test_files_das,
)

print("CAS duration bin edges (33rd / 67th percentile, seconds):", cas_bins)
print("DAS duration bin edges (33rd / 67th percentile, seconds):", das_bins)
display(cas_table)
display(das_table)

CAS duration bin edges (33rd / 67th percentile, seconds): (np.float64(0.608), np.float64(0.896))
DAS duration bin edges (33rd / 67th percentile, seconds): (np.float64(0.768), np.float64(0.96))


,Task,Model,FP events/recording,Short (<0.61s) recall (%),Short (<0.61s) n,Medium (0.61-0.90s) recall (%),Medium (0.61-0.90s) n,Long (>=0.90s) recall (%),Long (>=0.90s) n
0,CAS,Standard BCE,3.16,32.9,765,53.6,811,52.8,808
1,CAS,Raw weighted BCE,3.52,33.2,765,55.2,811,59.3,808


,Task,Model,FP events/recording,Short (<0.77s) recall (%),Short (<0.77s) n,Medium (0.77-0.96s) recall (%),Medium (0.77-0.96s) n,Long (>=0.96s) recall (%),Long (>=0.96s) n
0,DAS,Standard BCE,2.92,53.5,510,69.7,656,70.4,626
1,DAS,Raw weighted BCE,2.88,53.3,510,65.2,656,63.1,626
